<a href="https://colab.research.google.com/github/MO230101/Copolymer-lipid-interaction-study_ver.3/blob/main/03_SolutionNMR_endpoint_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 03_SolutionNMR_endpoint_generation.py
# ============================================================
#
# PURPOSE
# -------
# Reconstruct the Solution-NMR endpoint pipeline used by the
# established analysis while preserving the original
# WithinRegime_Z values exactly.
#
# INPUTS
# ------
# 1) roi_peak_features_with_blank_relative.csv
# 2) ZG_to_FunctionalGroup_weights.csv
# 3) 06_Hierarchical_Language_Representation.csv
#
# IMPORTANT
# ---------
# The original algorithm that generated WithinRegime_Z from
# the motif scores is not re-estimated here.
#
# Instead:
#   Part A reproduces the old Code10 ROI -> motif calculation
#          for provenance / audit.
#
#   Part B extracts the established WithinRegime_Z endpoints
#          from 06_Hierarchical_Language_Representation.csv.
#
# This avoids silently replacing the historical hierarchical
# normalization with a newly invented global Z-score.
#
# FINAL OUTPUT FOR CODE05
# -----------------------
# 03_SolutionNMR_independent_Y.csv
#
# ============================================================


# ============================================================
# 0. IMPORT
# ============================================================

import os
import re
import json
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. SETTINGS
# ============================================================

EXPECTED_N = 43

PRIMARY_RESPONSE = "delta_height"

OUTPUT_DIR = Path(
    "03_SolutionNMR_endpoint_generation_FINAL"
)

ZIP_PATH = Path(
    "03_SolutionNMR_endpoint_generation_FINAL.zip"
)

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()


# ============================================================
# 2. FINAL HISTORICAL Y COLUMNS
# ============================================================

HISTORICAL_Y = {

    "Alkenyl_Response":
        "alkenyl_ch_WithinRegime_Z",

    "AlkylChain_Response":
        "alkyl_chain_chx_WithinRegime_Z",

    "Glycerol_Response":
        "glycerol_oxygenated_ch_WithinRegime_Z",

    "Other_Response":
        "other_ambiguous_WithinRegime_Z",
}


# ============================================================
# 3. CODE10 MOTIF DEFINITIONS
# ============================================================
#
# These are used for audit of the ROI -> motif processing.
#
# NOTE:
# Code10 used headgroup / glycerol / alkenyl / alkyl-chain
# motif scores.
#
# "Other" is a later hierarchical endpoint and is therefore
# NOT falsely reconstructed here as a Code10 motif.
#
# ============================================================

CODE10_TARGET_GROUPS = {

    "headgroup":
        "headgroup_polar_ch",

    "glycerol":
        "glycerol_oxygenated_ch",

    "alkenyl":
        "alkenyl_ch",

    "alkyl_chain":
        "alkyl_chain_chx",
}


# ============================================================
# 4. GENERIC HELPERS
# ============================================================

def robust_read_csv(path):

    last_error = None

    for encoding in [
        "utf-8-sig",
        "utf-8",
        "cp932",
        "latin1",
    ]:

        try:
            return pd.read_csv(
                path,
                encoding=encoding
            )

        except Exception as e:
            last_error = e

    raise last_error


def normalize_text(x):

    if pd.isna(x):
        return ""

    return str(x).strip()


def normalize_colname(x):

    return re.sub(
        r"[^a-z0-9]",
        "",
        str(x).lower()
    )


def canonical_id(x):

    x = normalize_text(x)

    x = re.sub(
        r"\s+",
        "",
        x
    )

    return x.upper()


def detect_column(
    df,
    candidates,
    required=True
):

    normalized = {
        normalize_colname(c): c
        for c in df.columns
    }

    for candidate in candidates:

        key = normalize_colname(
            candidate
        )

        if key in normalized:
            return normalized[key]

    if required:

        raise KeyError(
            "Could not detect required column.\n"
            f"Candidates: {candidates}\n"
            f"Available: {list(df.columns)}"
        )

    return None


# ============================================================
# 5. UPLOAD
# ============================================================

try:

    from google.colab import files

    print("=" * 80)
    print("Upload exactly these three files:")
    print()
    print("1) roi_peak_features_with_blank_relative.csv")
    print("2) ZG_to_FunctionalGroup_weights.csv")
    print("3) 06_Hierarchical_Language_Representation.csv")
    print("=" * 80)

    uploaded = files.upload()

    uploaded_paths = [
        Path(name)
        for name in uploaded.keys()
    ]

except ImportError:

    uploaded_paths = list(
        Path(".").glob("*.csv")
    )


# ============================================================
# 6. IDENTIFY FILES
# ============================================================

def find_input(fragment):

    matches = [
        p
        for p in uploaded_paths
        if fragment.lower() in p.name.lower()
    ]

    if not matches:

        raise FileNotFoundError(
            f"Input not found: {fragment}"
        )

    return matches[0]


ROI_FILE = find_input(
    "roi_peak_features_with_blank_relative"
)

WEIGHT_FILE = find_input(
    "ZG_to_FunctionalGroup_weights"
)

HIERARCHICAL_FILE = find_input(
    "06_Hierarchical_Language_Representation"
)


print("\nINPUTS")
print("-" * 80)
print("ROI         :", ROI_FILE.name)
print("Weights     :", WEIGHT_FILE.name)
print("Hierarchical:", HIERARCHICAL_FILE.name)


# ============================================================
# 7. LOAD
# ============================================================

roi = robust_read_csv(
    ROI_FILE
)

weights = robust_read_csv(
    WEIGHT_FILE
)

hier = robust_read_csv(
    HIERARCHICAL_FILE
)


roi.columns = [
    str(c).strip()
    for c in roi.columns
]

weights.columns = [
    str(c).strip()
    for c in weights.columns
]

hier.columns = [
    str(c).strip()
    for c in hier.columns
]


# ============================================================
# 8. DETECT ROI TABLE COLUMNS
# ============================================================

ROI_SAMPLE_COL = detect_column(
    roi,
    [
        "Sample_Name",
        "Sample",
        "Copolymer_Name",
        "Canonical_ID",
    ]
)

ROI_LABEL_COL = detect_column(
    roi,
    [
        "ROI_Label",
        "ROI",
        "ZG_ROI",
    ]
)

DELTA_HEIGHT_COL = detect_column(
    roi,
    [
        "delta_height",
    ]
)

DELTA_AUC_COL = detect_column(
    roi,
    [
        "delta_auc",
    ],
    required=False
)


# ============================================================
# 9. DETECT WEIGHT TABLE COLUMNS
# ============================================================

WEIGHT_ROI_COL = detect_column(
    weights,
    [
        "ZG_ROI",
        "ROI_Label",
        "ROI",
    ]
)

FUNCTIONAL_GROUP_COL = detect_column(
    weights,
    [
        "FunctionalGroup",
        "Functional_Group",
    ]
)

WEIGHT_COL = detect_column(
    weights,
    [
        "weight",
    ]
)


# ============================================================
# 10. DETECT HIERARCHICAL ID
# ============================================================

HIER_ID_COL = detect_column(
    hier,
    [
        "Canonical_ID",
        "Sample_Key",
        "Sample_Name",
        "Copolymer_Name",
        "Sample",
    ]
)


# ============================================================
# 11. STANDARDIZE IDENTIFIERS
# ============================================================

roi = roi.copy()

roi["Canonical_ID"] = (
    roi[ROI_SAMPLE_COL]
    .map(canonical_id)
)


hier = hier.copy()

hier["Canonical_ID"] = (
    hier[HIER_ID_COL]
    .map(canonical_id)
)


roi["ROI_Label_STD"] = (
    roi[ROI_LABEL_COL]
    .astype(str)
    .str.strip()
)


weights = weights.copy()

weights["ROI_Label_STD"] = (
    weights[WEIGHT_ROI_COL]
    .astype(str)
    .str.strip()
)


weights["FunctionalGroup_STD"] = (
    weights[FUNCTIONAL_GROUP_COL]
    .astype(str)
    .str.strip()
)


# ============================================================
# 12. NUMERIC CONVERSION
# ============================================================

roi["delta_height_numeric"] = pd.to_numeric(
    roi[DELTA_HEIGHT_COL],
    errors="coerce"
)


if DELTA_AUC_COL is not None:

    roi["delta_auc_numeric"] = pd.to_numeric(
        roi[DELTA_AUC_COL],
        errors="coerce"
    )

else:

    roi["delta_auc_numeric"] = np.nan


weights["weight_numeric"] = pd.to_numeric(
    weights[WEIGHT_COL],
    errors="coerce"
)


# ============================================================
# 13. QC — MAPPING WEIGHTS
# ============================================================

if weights["weight_numeric"].isna().any():

    bad = weights.loc[
        weights["weight_numeric"].isna()
    ]

    raise ValueError(
        "Non-numeric/missing mapping weights detected:\n"
        +
        bad.to_string(index=False)
    )


if (
    weights["weight_numeric"] < 0
).any():

    raise ValueError(
        "Negative mapping weights detected."
    )


# ============================================================
# 14. PRESERVE ORIGINAL WEIGHTS
# ============================================================
#
# CRITICAL:
#
# DO NOT normalize weights by functional group.
#
# The historical mapping already contains the established
# assignment weights.
#
# ============================================================

mapping_original = weights[
    [
        "ROI_Label_STD",
        "FunctionalGroup_STD",
        "weight_numeric",
    ]
].copy()


mapping_original.columns = [
    "ZG_ROI",
    "FunctionalGroup",
    "weight",
]


mapping_original = (
    mapping_original
    .sort_values(
        [
            "ZG_ROI",
            "FunctionalGroup",
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 15. AUDIT PER-ROI WEIGHT SUM
# ============================================================
#
# Historical mapping was normalized at the ZG-ROI level.
# Therefore sums close to 1 are expected for mapped ROIs.
#
# We report but do not alter them.
#
# ============================================================

weight_sum_audit = (

    mapping_original

    .groupby(
        "ZG_ROI",
        as_index=False
    )["weight"]

    .sum()

    .rename(
        columns={
            "weight":
                "Sum_of_original_weights"
        }
    )
)


weight_sum_audit[
    "Close_to_1"
] = np.isclose(
    weight_sum_audit[
        "Sum_of_original_weights"
    ],
    1.0,
    atol=1e-6
)


# ============================================================
# 16. CODE10 SUPPRESSION RESPONSES
# ============================================================
#
# Old Code10:
#
# suppression = -delta
# negative values after inversion are clipped to zero.
#
# Missing measurements remain missing.
#
# ============================================================

roi["suppression_height"] = np.where(

    roi[
        "delta_height_numeric"
    ].notna(),

    np.clip(
        -roi[
            "delta_height_numeric"
        ],
        0.0,
        None
    ),

    np.nan
)


roi["suppression_auc"] = np.where(

    roi[
        "delta_auc_numeric"
    ].notna(),

    np.clip(
        -roi[
            "delta_auc_numeric"
        ],
        0.0,
        None
    ),

    np.nan
)


# ============================================================
# 17. CODE10 ROI × WEIGHT MERGE
# ============================================================

roi_weighted = roi.merge(

    mapping_original,

    left_on="ROI_Label_STD",

    right_on="ZG_ROI",

    how="left"
)


roi_weighted[
    "weighted_suppression_height"
] = (

    roi_weighted[
        "suppression_height"
    ]

    *

    roi_weighted[
        "weight"
    ]
)


roi_weighted[
    "weighted_suppression_auc"
] = (

    roi_weighted[
        "suppression_auc"
    ]

    *

    roi_weighted[
        "weight"
    ]
)


# ============================================================
# 18. CODE10 MOTIF SCORE RECONSTRUCTION
# ============================================================
#
# No within-material weight renormalization.
#
# If an ROI response is missing, it contributes no measured
# response. We do NOT redistribute its mapping weight to the
# remaining ROIs.
#
# ============================================================

code10_rows = []


all_samples = sorted(
    roi[
        "Canonical_ID"
    ]
    .dropna()
    .unique()
)


for sample in all_samples:

    sample_df = roi_weighted.loc[
        roi_weighted[
            "Canonical_ID"
        ]
        ==
        sample
    ]


    record = {
        "Canonical_ID":
            sample
    }


    for motif_name, fg_name in (
        CODE10_TARGET_GROUPS.items()
    ):

        sub = sample_df.loc[
            sample_df[
                "FunctionalGroup"
            ]
            ==
            fg_name
        ]


        valid_height = sub.loc[
            sub[
                "weighted_suppression_height"
            ].notna()
        ]


        if len(valid_height) > 0:

            height_score = float(
                valid_height[
                    "weighted_suppression_height"
                ].sum()
            )

        else:

            height_score = np.nan


        valid_auc = sub.loc[
            sub[
                "weighted_suppression_auc"
            ].notna()
        ]


        if len(valid_auc) > 0:

            auc_score = float(
                valid_auc[
                    "weighted_suppression_auc"
                ].sum()
            )

        else:

            auc_score = np.nan


        record[
            f"{motif_name}_height_score"
        ] = height_score


        record[
            f"{motif_name}_auc_score"
        ] = auc_score


        record[
            f"{motif_name}_n_contributing_ROIs_height"
        ] = int(
            len(
                valid_height
            )
        )


    code10_rows.append(
        record
    )


code10_scores = pd.DataFrame(
    code10_rows
)


# ============================================================
# 19. WITHIN-SAMPLE NORMALIZED CODE10 MOTIF SCORES
# ============================================================
#
# This reproduces the descriptive relative motif preference
# step without redefining the final Y.
#
# ============================================================

height_cols = [
    f"{m}_height_score"
    for m in CODE10_TARGET_GROUPS
]


def normalize_row_nonnegative(
    row,
    columns
):

    values = (
        pd.to_numeric(
            row[columns],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    valid = np.isfinite(
        values
    )


    total = np.nansum(
        values
    )


    out = np.full(
        len(values),
        np.nan
    )


    if (
        valid.any()
        and
        total > 0
    ):

        out[valid] = (
            values[valid]
            /
            total
        )


    return out


normalized_matrix = []


for _, row in code10_scores.iterrows():

    normalized_matrix.append(
        normalize_row_nonnegative(
            row,
            height_cols
        )
    )


normalized_matrix = np.asarray(
    normalized_matrix
)


for j, motif in enumerate(
    CODE10_TARGET_GROUPS
):

    code10_scores[
        f"{motif}_height_relative"
    ] = normalized_matrix[
        :,
        j
    ]


# ============================================================
# 20. VALIDATE HISTORICAL WITHINREGIME_Z
# ============================================================

missing_y = [

    source_col

    for source_col in HISTORICAL_Y.values()

    if source_col not in hier.columns
]


if missing_y:

    print("\nAvailable hierarchical columns:")

    for c in hier.columns:
        print(" -", c)

    raise ValueError(
        "\nThe historical hierarchical table does not "
        "contain all required WithinRegime_Z endpoints:\n"
        +
        "\n".join(
            missing_y
        )
        +
        "\n\nDo NOT replace these columns with a new "
        "global Z-score. The historical hierarchical "
        "generation code would be required to regenerate "
        "them exactly."
    )


# ============================================================
# 21. NUMERIC VALIDATION OF FINAL Y
# ============================================================

for source_col in HISTORICAL_Y.values():

    hier[source_col] = pd.to_numeric(
        hier[source_col],
        errors="coerce"
    )


# ============================================================
# 22. EXTRACT FINAL INDEPENDENT Y
# ============================================================

final_y = hier[
    [
        "Canonical_ID",
        *HISTORICAL_Y.values(),
    ]
].copy()


# Friendly aliases for Code05
for alias, source_col in (
    HISTORICAL_Y.items()
):

    final_y[
        alias
    ] = final_y[
        source_col
    ]


# ============================================================
# 23. REMOVE EXACT DUPLICATES ONLY
# ============================================================

if final_y[
    "Canonical_ID"
].duplicated().any():

    duplicated_ids = final_y.loc[
        final_y[
            "Canonical_ID"
        ].duplicated(
            keep=False
        ),
        "Canonical_ID"
    ].unique()


    for material_id in duplicated_ids:

        sub = final_y.loc[
            final_y[
                "Canonical_ID"
            ]
            ==
            material_id
        ]


        value_cols = list(
            HISTORICAL_Y.values()
        )


        unique_values = (
            sub[
                value_cols
            ]
            .drop_duplicates()
        )


        if len(unique_values) > 1:

            raise ValueError(
                "Conflicting duplicate historical Y "
                f"rows for {material_id}."
            )


    final_y = (
        final_y
        .drop_duplicates(
            subset=[
                "Canonical_ID"
            ],
            keep="first"
        )
    )


# ============================================================
# 24. SORT
# ============================================================

final_y = (

    final_y

    .sort_values(
        "Canonical_ID"
    )

    .reset_index(
        drop=True
    )
)


# ============================================================
# 25. FINAL Y QC
# ============================================================

historical_cols = list(
    HISTORICAL_Y.values()
)


missing_by_col = (
    final_y[
        historical_cols
    ]
    .isna()
    .sum()
)


if (
    missing_by_col > 0
).any():

    print(
        "\nWARNING: missing historical Y values:"
    )

    print(
        missing_by_col[
            missing_by_col > 0
        ]
    )


if len(final_y) != EXPECTED_N:

    print(
        f"\nWARNING: expected {EXPECTED_N} materials "
        f"but hierarchical table produced "
        f"{len(final_y)} unique materials."
    )


# ============================================================
# 26. CROSS-CHECK IDs BETWEEN CODE10 AND FINAL Y
# ============================================================

code10_ids = set(
    code10_scores[
        "Canonical_ID"
    ]
)


final_y_ids = set(
    final_y[
        "Canonical_ID"
    ]
)


id_audit_rows = []


for material_id in sorted(
    code10_ids
    |
    final_y_ids
):

    id_audit_rows.append(
        {
            "Canonical_ID":
                material_id,

            "Present_in_Code10_ROI_pipeline":
                material_id in code10_ids,

            "Present_in_Historical_Hierarchical_Y":
                material_id in final_y_ids,
        }
    )


id_audit = pd.DataFrame(
    id_audit_rows
)


# ============================================================
# 27. JOIN CODE10 AUDIT WITH FINAL Y
# ============================================================

code10_vs_y = final_y.merge(

    code10_scores,

    on="Canonical_ID",

    how="left"
)


# ============================================================
# 28. FINAL OUTPUT — THIS IS THE FILE USED BY CODE05
# ============================================================

FINAL_Y_PATH = (

    OUTPUT_DIR
    /
    "03_SolutionNMR_independent_Y.csv"
)


final_y.to_csv(
    FINAL_Y_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 29. SAVE CODE10 AUDIT TABLES
# ============================================================

ROI_PROCESSED_PATH = (
    OUTPUT_DIR
    /
    "01_Code10_ROI_processed.csv"
)


roi_weighted.to_csv(
    ROI_PROCESSED_PATH,
    index=False,
    encoding="utf-8-sig"
)


CODE10_SCORE_PATH = (
    OUTPUT_DIR
    /
    "02_Code10_motif_scores_reconstructed.csv"
)


code10_scores.to_csv(
    CODE10_SCORE_PATH,
    index=False,
    encoding="utf-8-sig"
)


MAPPING_PATH = (
    OUTPUT_DIR
    /
    "04_ROI_to_FunctionalGroup_original_weights.csv"
)


mapping_original.to_csv(
    MAPPING_PATH,
    index=False,
    encoding="utf-8-sig"
)


WEIGHT_AUDIT_PATH = (
    OUTPUT_DIR
    /
    "05_ROI_weight_sum_audit.csv"
)


weight_sum_audit.to_csv(
    WEIGHT_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig"
)


ID_AUDIT_PATH = (
    OUTPUT_DIR
    /
    "06_Material_ID_audit.csv"
)


id_audit.to_csv(
    ID_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig"
)


CODE10_VS_Y_PATH = (
    OUTPUT_DIR
    /
    "07_Code10_vs_FinalY_audit.csv"
)


code10_vs_y.to_csv(
    CODE10_VS_Y_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 30. ROI MAPPING FIGURE
# ============================================================
#
# IMPORTANT:
#
# Use the full mapping table, not only ROIs present in the
# 1D response table.
#
# Therefore ROI.15 remains visible when present in the
# historical mapping even if it was excluded from a specific
# 1D quantification step.
#
# ============================================================

DISPLAY_GROUPS = {

    "glycerol_oxygenated_ch":
        "Glycerol / oxygenated CH",

    "alkenyl_ch":
        "Alkenyl CH",

    "alkyl_chain_chx":
        "Alkyl-chain CHx",

    "other_ambiguous":
        "Other / ambiguous",
}


mapping_display = mapping_original.loc[
    mapping_original[
        "FunctionalGroup"
    ].isin(
        DISPLAY_GROUPS.keys()
    )
].copy()


if len(mapping_display) > 0:

    heatmap = mapping_display.pivot_table(

        index="FunctionalGroup",

        columns="ZG_ROI",

        values="weight",

        aggfunc="sum",

        fill_value=0.0
    )


    # Natural ROI order
    def roi_number(x):

        match = re.search(
            r"(\d+)",
            str(x)
        )

        if match:
            return int(
                match.group(1)
            )

        return 9999


    ordered_rois = sorted(
        heatmap.columns,
        key=roi_number
    )


    heatmap = heatmap[
        ordered_rois
    ]


    display_order = [
        fg
        for fg in DISPLAY_GROUPS
        if fg in heatmap.index
    ]


    heatmap = heatmap.loc[
        display_order
    ]


    fig_width = max(
        9.0,
        0.65
        *
        len(
            heatmap.columns
        )
    )


    fig, ax = plt.subplots(
        figsize=(
            fig_width,
            3.6
        )
    )


    im = ax.imshow(
        heatmap.to_numpy(),
        aspect="auto",
        vmin=0,
        vmax=1
    )


    ax.set_xticks(
        np.arange(
            len(
                heatmap.columns
            )
        )
    )


    ax.set_xticklabels(
        heatmap.columns,
        rotation=45,
        ha="right"
    )


    ax.set_yticks(
        np.arange(
            len(
                heatmap.index
            )
        )
    )


    ax.set_yticklabels(
        [
            DISPLAY_GROUPS[x]
            for x in heatmap.index
        ]
    )


    for i in range(
        heatmap.shape[0]
    ):

        for j in range(
            heatmap.shape[1]
        ):

            value = float(
                heatmap.iloc[
                    i,
                    j
                ]
            )

            if value > 0:

                ax.text(
                    j,
                    i,
                    f"{value:.2f}",
                    ha="center",
                    va="center",
                    fontsize=8
                )


    cbar = fig.colorbar(
        im,
        ax=ax
    )

    cbar.set_label(
        "Original assignment weight"
    )


    ax.set_xlabel(
        "Solution-NMR ROI"
    )

    ax.set_ylabel(
        ""
    )


    ax.set_title(
        "Mapping of NMR ROIs to lipid-associated regions"
    )


    plt.tight_layout()


    FIG_PNG = (
        OUTPUT_DIR
        /
        "FigS_ROI_to_LipidRegion_Mapping.png"
    )


    FIG_PDF = (
        OUTPUT_DIR
        /
        "FigS_ROI_to_LipidRegion_Mapping.pdf"
    )


    plt.savefig(
        FIG_PNG,
        dpi=600,
        bbox_inches="tight"
    )


    plt.savefig(
        FIG_PDF,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )

else:

    FIG_PNG = None
    FIG_PDF = None


# ============================================================
# 31. SOURCE-DATA WORKBOOK
# ============================================================

SOURCE_XLSX = (
    OUTPUT_DIR
    /
    "08_SolutionNMR_endpoint_source_data.xlsx"
)


with pd.ExcelWriter(
    SOURCE_XLSX,
    engine="openpyxl"
) as writer:


    final_y.to_excel(
        writer,
        sheet_name="Final_independent_Y",
        index=False
    )


    code10_scores.to_excel(
        writer,
        sheet_name="Code10_motif_scores",
        index=False
    )


    mapping_original.to_excel(
        writer,
        sheet_name="Original_ROI_mapping",
        index=False
    )


    weight_sum_audit.to_excel(
        writer,
        sheet_name="ROI_weight_audit",
        index=False
    )


    id_audit.to_excel(
        writer,
        sheet_name="Material_ID_audit",
        index=False
    )


    code10_vs_y.to_excel(
        writer,
        sheet_name="Code10_vs_FinalY",
        index=False
    )


# ============================================================
# 32. METADATA
# ============================================================

metadata = {

    "pipeline":
        "03_SolutionNMR_endpoint_generation_FINAL",

    "role_in_final_analysis":
        (
            "Independent experimental Y only; "
            "not used to construct or select X."
        ),

    "inputs": [
        ROI_FILE.name,
        WEIGHT_FILE.name,
        HIERARCHICAL_FILE.name,
    ],

    "Code10_primary_response":
        PRIMARY_RESPONSE,

    "Code10_suppression_definition":
        "max(-delta_height, 0)",

    "Code10_weight_rule":
        (
            "Original ZG_ROI-to-functional-group "
            "assignment weights; no new "
            "functional-group normalization."
        ),

    "missing_ROI_rule":
        (
            "No redistribution or renormalization "
            "of mapping weights."
        ),

    "final_Y_source":
        HIERARCHICAL_FILE.name,

    "final_Y_columns":
        list(
            HISTORICAL_Y.values()
        ),

    "new_global_zscore_calculated":
        False,

    "WithinRegime_Z_reestimated":
        False,

    "reason":
        (
            "The original hierarchical-generation "
            "algorithm was not recovered from the "
            "available historical source code. "
            "Established WithinRegime_Z values are "
            "therefore preserved exactly rather than "
            "replaced by an inferred transformation."
        ),

    "SolutionNMR_used_as_X":
        False,

    "SolutionNMR_used_for_selection":
        False,

    "SolutionNMR_used_as_independent_Y":
        True,

    "expected_materials":
        EXPECTED_N,
}


METADATA_PATH = (
    OUTPUT_DIR
    /
    "09_Metadata.json"
)


with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 33. README
# ============================================================

README_PATH = (
    OUTPUT_DIR
    /
    "README.txt"
)


readme = """
03 Solution-NMR independent endpoint generation

This script has two distinct purposes.

PART A
------
Reconstruct the historical Code10 ROI-to-motif calculation:

delta_height
    -> suppression = max(-delta_height, 0)
    -> original ROI-to-functional-group assignment weights
    -> weighted motif responses

The original mapping weights are preserved.
They are NOT re-normalized by functional group.
Missing ROI responses do NOT cause redistribution of weights.

PART B
------
Generate the final independent Solution-NMR Y table used by
the final exploration analysis.

The final endpoints are extracted from the established
06_Hierarchical_Language_Representation.csv:

alkenyl_ch_WithinRegime_Z
alkyl_chain_chx_WithinRegime_Z
glycerol_oxygenated_ch_WithinRegime_Z
other_ambiguous_WithinRegime_Z

The historical algorithm that originally generated
WithinRegime_Z was not recovered from the available source
code. Therefore this script deliberately does NOT invent or
re-estimate that transformation.

The exact established endpoint values are preserved.

Final file used by Code05:

03_SolutionNMR_independent_Y.csv

Solution-NMR is an independent evaluation endpoint and is
not used for X construction or material selection.
"""


with open(
    README_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        readme
    )


# ============================================================
# 34. FINAL QC
# ============================================================

print("\n" + "=" * 80)
print("03 FINAL SOLUTION-NMR ENDPOINT GENERATION")
print("=" * 80)


print(
    "\nCode10 ROI materials:",
    code10_scores[
        "Canonical_ID"
    ].nunique()
)


print(
    "Final historical Y materials:",
    final_y[
        "Canonical_ID"
    ].nunique()
)


print(
    "\nFinal Y columns:"
)


for alias, source in (
    HISTORICAL_Y.items()
):

    print(
        f"  {alias:22s} <- {source}"
    )


print(
    "\nMissing final Y:"
)


print(
    final_y[
        historical_cols
    ]
    .isna()
    .sum()
    .to_string()
)


print(
    "\nFinal output for Code05:"
)


print(
    FINAL_Y_PATH
)


print(
    "\nIMPORTANT:"
)


print(
    "No new global Z-score was calculated."
)


print(
    "No historical WithinRegime_Z value was recalculated."
)


# ============================================================
# 35. ZIP
# ============================================================

output_files = [

    ROI_PROCESSED_PATH,

    CODE10_SCORE_PATH,

    FINAL_Y_PATH,

    MAPPING_PATH,

    WEIGHT_AUDIT_PATH,

    ID_AUDIT_PATH,

    CODE10_VS_Y_PATH,

    SOURCE_XLSX,

    METADATA_PATH,

    README_PATH,
]


if FIG_PNG is not None:

    output_files.extend(
        [
            FIG_PNG,
            FIG_PDF,
        ]
    )


with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for path in output_files:

        z.write(
            path,
            arcname=path.name
        )


print(
    "\nZIP generated:"
)


print(
    ZIP_PATH
)


# ============================================================
# 36. DOWNLOAD
# ============================================================

try:

    from google.colab import files

    files.download(
        str(
            ZIP_PATH
        )
    )

except ImportError:

    pass

Upload exactly these three files:

1) roi_peak_features_with_blank_relative.csv
2) ZG_to_FunctionalGroup_weights.csv
3) 06_Hierarchical_Language_Representation.csv


Saving 06_Hierarchical_Language_Representation.csv to 06_Hierarchical_Language_Representation.csv
Saving roi_peak_features_with_blank_relative.csv to roi_peak_features_with_blank_relative (3).csv
Saving ZG_to_FunctionalGroup_weights.csv to ZG_to_FunctionalGroup_weights (3).csv

INPUTS
--------------------------------------------------------------------------------
ROI         : roi_peak_features_with_blank_relative (3).csv
Weights     : ZG_to_FunctionalGroup_weights (3).csv
Hierarchical: 06_Hierarchical_Language_Representation.csv

03 FINAL SOLUTION-NMR ENDPOINT GENERATION

Code10 ROI materials: 44
Final historical Y materials: 43

Final Y columns:
  Alkenyl_Response       <- alkenyl_ch_WithinRegime_Z
  AlkylChain_Response    <- alkyl_chain_chx_WithinRegime_Z
  Glycerol_Response      <- glycerol_oxygenated_ch_WithinRegime_Z
  Other_Response         <- other_ambiguous_WithinRegime_Z

Missing final Y:
alkenyl_ch_WithinRegime_Z                0
alkyl_chain_chx_WithinRegime_Z           0
g

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>